# SignSync: ASL Alphabet Recognition

A deep neural network built from scratch to recognize American Sign Language (ASL) alphabet gestures.

**Model:** 784 → 128 → 64 → 24 (Dense NN)  
**Test Accuracy:** ~78%  
**GPU Accelerated:** Uses CuPy when available

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inshaal81/SignSync/blob/main/notebooks/SignSync_Colab.ipynb)

## 1. Setup & GPU Detection

In [ ]:
# Install CuPy for GPU acceleration
!pip install cupy-cuda12x -q

import numpy as np

# Try to use CuPy (GPU), fall back to NumPy (CPU)
try:
    import cupy as cp
    xp = cp
    GPU_AVAILABLE = True
    print("GPU detected! Using CuPy for acceleration.")
    print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
except:
    xp = np
    GPU_AVAILABLE = False
    print("No GPU detected. Using NumPy (CPU).")

def to_numpy(arr):
    """Convert array to numpy (for visualization)."""
    if GPU_AVAILABLE and hasattr(arr, 'get'):
        return arr.get()
    return arr

## 2. Upload Dataset

Download the Sign Language MNIST dataset from Kaggle:
https://www.kaggle.com/datasets/datamunge/sign-language-mnist

Upload `sign_mnist_train.csv` and `sign_mnist_test.csv` when prompted.

In [ ]:
from google.colab import files
import pandas as pd

print("Upload sign_mnist_train.csv and sign_mnist_test.csv")
uploaded = files.upload()

In [ ]:
# Load the uploaded CSV files
train_df = pd.read_csv('sign_mnist_train.csv')
test_df = pd.read_csv('sign_mnist_test.csv')

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Features: {train_df.shape[1] - 1} pixels (28x28 images)")

## 3. Data Preprocessing

In [ ]:
def load_and_preprocess_data(train_df, test_df):
    """Load and preprocess ASL alphabet dataset."""
    
    # Separate features and labels (use numpy first, then convert)
    train_x = train_df.drop('label', axis=1).values.T.astype(np.float32)
    train_y_raw = train_df['label'].values
    
    test_x = test_df.drop('label', axis=1).values.T.astype(np.float32)
    test_y_raw = test_df['label'].values
    
    # Remove J (9) and Z (25) - they require motion
    train_mask = (train_y_raw != 9) & (train_y_raw != 25)
    train_x = train_x[:, train_mask]
    train_y_raw = train_y_raw[train_mask]
    
    test_mask = (test_y_raw != 9) & (test_y_raw != 25)
    test_x = test_x[:, test_mask]
    test_y_raw = test_y_raw[test_mask]
    
    # Remap labels: 0-8 stay same, 10-24 become 9-23
    def remap_labels(labels):
        remapped = labels.copy()
        remapped[labels > 9] -= 1
        return remapped
    
    train_y = remap_labels(train_y_raw).reshape(1, -1)
    test_y = remap_labels(test_y_raw).reshape(1, -1)
    
    # Per-image normalization (key for handling distribution shift)
    def normalize_per_image(X):
        means = np.mean(X, axis=0, keepdims=True)
        stds = np.std(X, axis=0, keepdims=True) + 1e-8
        return (X - means) / stds
    
    train_x = normalize_per_image(train_x)
    test_x = normalize_per_image(test_x)
    
    # Convert to GPU arrays if available
    train_x = xp.asarray(train_x)
    train_y = xp.asarray(train_y)
    test_x = xp.asarray(test_x)
    test_y = xp.asarray(test_y)
    
    # Class names (A-Z minus J and Z)
    classes = np.array([chr(i) for i in range(65, 91) if chr(i) not in ['J', 'Z']])
    
    return train_x, train_y, test_x, test_y, classes

# Load data
train_x, train_y, test_x, test_y, classes = load_and_preprocess_data(train_df, test_df)

print(f"\nProcessed data:")
print(f"  Train X shape: {train_x.shape}")
print(f"  Train Y shape: {train_y.shape}")
print(f"  Test X shape: {test_x.shape}")
print(f"  Test Y shape: {test_y.shape}")
print(f"  Classes: {len(classes)} ({classes[0]}-{classes[-1]}, excluding J, Z)")
print(f"  Array type: {type(train_x).__module__.split('.')[0]}")

## 4. Visualize Sample Data

In [ ]:
import matplotlib.pyplot as plt

# Display sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
train_x_np = to_numpy(train_x)
train_y_np = to_numpy(train_y)

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, train_x_np.shape[1])
    img = train_x_np[:, idx].reshape(28, 28)
    label = int(train_y_np[0, idx])
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: {classes[label]}")
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Neural Network Implementation (GPU-Accelerated)

In [ ]:
# Activation functions (using xp for GPU/CPU compatibility)
def relu(Z):
    A = xp.maximum(0, Z)
    return A, Z

def softmax(Z):
    exp_Z = xp.exp(Z - xp.max(Z, axis=0, keepdims=True))
    A = exp_Z / xp.sum(exp_Z, axis=0, keepdims=True)
    return A, Z

def relu_backward(dA, cache):
    Z = cache
    dZ = xp.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ

# Parameter initialization (He initialization)
def initialize_parameters_deep(layer_dims):
    xp.random.seed(1)
    parameters = {}
    L = len(layer_dims)
    
    for l in range(1, L):
        parameters['W' + str(l)] = xp.random.randn(layer_dims[l], layer_dims[l-1]).astype(xp.float32) / xp.sqrt(layer_dims[l-1])
        parameters['b' + str(l)] = xp.zeros((layer_dims[l], 1), dtype=xp.float32)
    
    return parameters

# Forward propagation
def linear_forward(A, W, b):
    Z = W.dot(A) + b
    cache = (A, W, b)
    return Z, cache

def linear_activation_forward(A_prev, W, b, activation):
    Z, linear_cache = linear_forward(A_prev, W, b)
    if activation == "relu":
        A, activation_cache = relu(Z)
    elif activation == "softmax":
        A, activation_cache = softmax(Z)
    cache = (linear_cache, activation_cache)
    return A, cache

def L_model_forward(X, parameters):
    caches = []
    A = X
    L = len(parameters) // 2
    
    for l in range(1, L):
        A_prev = A
        A, cache = linear_activation_forward(A_prev, parameters['W' + str(l)], parameters['b' + str(l)], "relu")
        caches.append(cache)
    
    AL, cache = linear_activation_forward(A, parameters['W' + str(L)], parameters['b' + str(L)], "softmax")
    caches.append(cache)
    
    return AL, caches

# Cost function
def compute_cost(AL, Y, parameters=None, lambd=0.0):
    m = Y.shape[1]
    Y_one_hot = xp.zeros((AL.shape[0], m), dtype=xp.float32)
    Y_one_hot[Y.astype(int), xp.arange(m)] = 1
    cost = -xp.sum(Y_one_hot * xp.log(AL + 1e-8)) / m
    
    if lambd > 0 and parameters is not None:
        L = len(parameters) // 2
        L2_cost = sum(xp.sum(xp.square(parameters['W' + str(l)])) for l in range(1, L+1))
        cost += (lambd / (2 * m)) * L2_cost
    
    return float(cost)

# Backward propagation
def linear_backward(dZ, cache, lambd=0.0):
    A_prev, W, b = cache
    m = A_prev.shape[1]
    dW = (1./m) * xp.dot(dZ, A_prev.T) + (lambd/m) * W
    db = (1./m) * xp.sum(dZ, axis=1, keepdims=True)
    dA_prev = xp.dot(W.T, dZ)
    return dA_prev, dW, db

def linear_activation_backward(dA, cache, activation, lambd=0.0):
    linear_cache, activation_cache = cache
    if activation == "relu":
        dZ = relu_backward(dA, activation_cache)
    dA_prev, dW, db = linear_backward(dZ, linear_cache, lambd=lambd)
    return dA_prev, dW, db

# Update parameters
def update_parameters(parameters, grads, learning_rate):
    L = len(parameters) // 2
    for l in range(L):
        parameters["W" + str(l+1)] -= learning_rate * grads["dW" + str(l+1)]
        parameters["b" + str(l+1)] -= learning_rate * grads["db" + str(l+1)]
    return parameters

print("Neural network functions defined!")
print(f"Using: {'CuPy (GPU)' if GPU_AVAILABLE else 'NumPy (CPU)'}")

In [ ]:
class DeepNeuralNetwork:
    """L-layer Deep Neural Network for multi-class classification (GPU-accelerated)."""
    
    def __init__(self, layer_dims, learning_rate=0.03, lambd=0.0):
        self.layer_dims = layer_dims
        self.learning_rate = learning_rate
        self.parameters = initialize_parameters_deep(layer_dims)
        self.costs = []
        self.lambd = lambd
    
    def train(self, X, Y, num_iterations=2500, print_cost=True):
        for i in range(num_iterations):
            # Forward propagation
            caches = []
            A = X
            L = len(self.parameters) // 2
            
            for l in range(1, L):
                A_prev = A
                A, cache = linear_activation_forward(
                    A_prev, self.parameters['W' + str(l)], 
                    self.parameters['b' + str(l)], "relu"
                )
                caches.append(cache)
            
            AL, cache = linear_activation_forward(
                A, self.parameters['W' + str(L)], 
                self.parameters['b' + str(L)], "softmax"
            )
            caches.append(cache)
            
            # Compute cost
            cost = compute_cost(AL, Y, self.parameters, lambd=self.lambd)
            
            # Backward propagation
            grads = {}
            m = AL.shape[1]
            Y_one_hot = xp.zeros_like(AL)
            Y_one_hot[Y.astype(int), xp.arange(m)] = 1
            
            dAL = AL - Y_one_hot
            current_cache = caches[L-1]
            linear_cache, _ = current_cache
            dA_prev_temp, dW_temp, db_temp = linear_backward(dAL, linear_cache, lambd=self.lambd)
            grads["dA" + str(L-1)] = dA_prev_temp
            grads["dW" + str(L)] = dW_temp
            grads["db" + str(L)] = db_temp
            
            for l in reversed(range(L-1)):
                current_cache = caches[l]
                dA_prev_temp, dW_temp, db_temp = linear_activation_backward(
                    grads["dA" + str(l + 1)], current_cache, "relu", lambd=self.lambd
                )
                grads["dA" + str(l)] = dA_prev_temp
                grads["dW" + str(l + 1)] = dW_temp
                grads["db" + str(l + 1)] = db_temp
            
            # Update parameters
            self.parameters = update_parameters(self.parameters, grads, self.learning_rate)
            
            # Record cost
            if i % 100 == 0:
                self.costs.append(cost)
                if print_cost and i % 500 == 0:
                    print(f"Cost after iteration {i}: {cost:.6f}")
        
        return self.parameters, self.costs
    
    def predict(self, X):
        AL, _ = L_model_forward(X, self.parameters)
        predictions = xp.argmax(AL, axis=0).reshape(1, -1)
        return predictions
    
    def evaluate(self, X, Y):
        predictions = self.predict(X)
        accuracy = float(xp.mean(predictions == Y))
        return accuracy

print("DeepNeuralNetwork class defined!")

## 6. Train the Model

In [ ]:
import time

# Model architecture
n_x = train_x.shape[0]  # 784 (28x28 pixels)
layer_dims = [n_x, 128, 64, 24]  # 24 output classes

print("="*50)
print("SignSync: ASL Alphabet Recognition")
print("="*50)
print(f"\nBackend: {'CuPy (GPU)' if GPU_AVAILABLE else 'NumPy (CPU)'}")
print(f"Architecture: {' -> '.join(map(str, layer_dims))}")
print(f"Total parameters: {sum(layer_dims[i]*layer_dims[i+1] + layer_dims[i+1] for i in range(len(layer_dims)-1)):,}")

# Create and train model
model = DeepNeuralNetwork(
    layer_dims, 
    learning_rate=0.03,  # Optimized learning rate
    lambd=0.0
)

print(f"\nTraining for 3000 iterations...\n")
start_time = time.time()
parameters, costs = model.train(train_x, train_y, num_iterations=3000, print_cost=True)
elapsed = time.time() - start_time

print(f"\nTraining completed in {elapsed:.2f} seconds")

## 7. Evaluate Results

In [ ]:
# Calculate accuracies
train_acc = model.evaluate(train_x, train_y)
test_acc = model.evaluate(test_x, test_y)

print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(f"Training Accuracy: {train_acc:.2%}")
print(f"Test Accuracy:     {test_acc:.2%}")
print("="*50)

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 6))
plt.plot(costs, 'b-', linewidth=2)
plt.ylabel('Cost', fontsize=12)
plt.xlabel('Iterations (x100)', fontsize=12)
plt.title(f'Training Progress (LR={model.learning_rate})', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

## 8. Test on Individual Samples

In [ ]:
# Visualize predictions on random test samples
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

test_x_np = to_numpy(test_x)
test_y_np = to_numpy(test_y)

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, test_x_np.shape[1])
    img = test_x_np[:, idx].reshape(28, 28)
    true_label = int(test_y_np[0, idx])
    
    # Get prediction
    sample = xp.asarray(test_x_np[:, idx:idx+1])
    pred_label = int(to_numpy(model.predict(sample))[0, 0])
    
    ax.imshow(img, cmap='gray')
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f"True: {classes[true_label]} | Pred: {classes[pred_label]}", 
                 color=color, fontsize=11)
    ax.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Per-Class Accuracy Analysis

In [ ]:
# Calculate per-class accuracy
predictions = to_numpy(model.predict(test_x))
test_y_np = to_numpy(test_y)

per_class_acc = []
for c in range(24):
    mask = test_y_np[0] == c
    if np.sum(mask) > 0:
        acc = np.mean(predictions[0, mask] == c)
        per_class_acc.append((classes[c], acc, np.sum(mask)))

# Sort by accuracy
per_class_acc.sort(key=lambda x: x[1], reverse=True)

print("Per-Class Accuracy (sorted):")
print("-" * 35)
for letter, acc, count in per_class_acc:
    bar = '#' * int(acc * 20)
    print(f"{letter}: {acc:6.1%} {bar}")

# Plot
plt.figure(figsize=(12, 6))
letters = [x[0] for x in per_class_acc]
accs = [x[1] for x in per_class_acc]
colors = ['green' if a > 0.8 else 'orange' if a > 0.6 else 'red' for a in accs]
plt.bar(letters, accs, color=colors)
plt.axhline(y=test_acc, color='blue', linestyle='--', label=f'Overall: {test_acc:.1%}')
plt.ylabel('Accuracy')
plt.xlabel('Letter')
plt.title('Per-Class Test Accuracy')
plt.legend()
plt.ylim(0, 1)
plt.show()

---

## Summary

This notebook demonstrates a GPU-accelerated neural network built from scratch for ASL alphabet recognition.

**Key Techniques:**
- CuPy for GPU acceleration (with NumPy fallback)
- He initialization for ReLU networks
- Per-image normalization to handle distribution shift
- Categorical cross-entropy loss with softmax
- Optimized learning rate (0.03)

**Results:**
- ~100% training accuracy
- ~78% test accuracy

For more details, visit: https://github.com/inshaal81/SignSync